# Fine-tune Vision-Language Model on Romanian VQA

Fine-tunes a 4-bit quantized vision-language model (Qwen2-VL / Gemma 3 / Llama 3.2 Vision)
on a Romanian translation of the GQA dataset using Unsloth + LoRA.

**Pipeline:**
1. Mount Drive and unzip GQA images
2. Load the Romanian-translated training JSON
3. Build a HuggingFace `Dataset` of `{messages: [...]}` conversations
4. Apply LoRA via Unsloth's `FastVisionModel`
5. Train with `SFTTrainer`
6. Save the LoRA adapter

**Hardware:** A100 40/80GB. Smaller GPUs need a smaller `per_device_train_batch_size`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
!pip install --no-deps --upgrade timm
import torch; torch._dynamo.config.recompile_limit = 64

In [ ]:
import json
import os
import random
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datasets import Dataset

## 2. Config

In [ ]:
# Paths
DATASET_DIR    = "/content/drive/MyDrive/Retranslation"
TRAIN_JSON     = "train_dataset_retranslated_gemma4_bigmodel.json"
IMAGES_ZIP     = "/content/drive/My Drive/licenta/images.zip"
IMAGES_EXTRACT = "/content/data"
IMAGE_DIR      = "/content/data/images"

# Output LoRA adapter
SAVE_DIR = "/content/drive/My Drive/licenta/new_llm/qwen-big-retranlsate-100k"

# Base model (4-bit pre-quantized for Unsloth)
BASE_MODEL = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"

# LoRA hyperparameters
LORA_R       = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0

# Training hyperparameters
PER_DEVICE_BS = 16
GRAD_ACCUM    = 8
MAX_STEPS     = 200
LEARNING_RATE = 2e-4

## 3. Unzip GQA images

In [ ]:
import zipfile
os.makedirs(IMAGES_EXTRACT, exist_ok=True)
with zipfile.ZipFile(IMAGES_ZIP, "r") as z:
    z.extractall(IMAGES_EXTRACT)

all_files   = os.listdir(IMAGE_DIR)
image_files = [f for f in all_files if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
print(f"Images extracted: {len(image_files)}")

## 4. Load training JSON

In [ ]:
def load_dataset(json_name):
    path = os.path.join(DATASET_DIR, json_name)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame.from_dict(data, orient="index")

train_df = load_dataset(TRAIN_JSON)
print(f"Training rows: {len(train_df)}")
train_df.head()

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
print(train_dataset[0])

## 5. Convert each row into a `messages` conversation

Each example becomes a user/assistant turn with a Romanian instruction, the image,
the question, and the gold answer. We pick a random instruction prompt per example
for light textual augmentation.

In [ ]:
# Romanian instruction prompts (the model is being trained on Romanian VQA,
# so these stay in Romanian on purpose).
INSTRUCTION_PROMPTS = [
    "Ești un asistent vizual care răspunde la întrebări despre imagine.",
    "Privește cu atenție imaginea și răspunde la întrebare.",
    "Analizează această imagine și oferă un răspuns la următoarea întrebare.",
    "Răspunde la întrebarea următoare pe baza imaginii furnizate.",
]

def preprocess(example):
    raw_id = (example.get('image_id') or example.get('imageId')
              or example.get('image') or example.get('id'))
    if raw_id is None:
        raise KeyError(f"Missing image id. Available keys: {list(example.keys())}")

    image_path = os.path.join(IMAGE_DIR, f"{raw_id}.jpg")
    image = Image.open(image_path).convert("RGB")

    instruction = random.choice(INSTRUCTION_PROMPTS)
    answer = example.get("answer_clean") or example.get("answer")

    conversation = [
        {"role": "user", "content": [
            {"type": "text",  "text": instruction},
            {"type": "image", "image": image},
            {"type": "text",  "text": f"Întrebare: {example['question']} Răspuns:"},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": answer},
        ]},
    ]
    return {"messages": conversation}

train_dataset = [preprocess(s) for s in tqdm(train_dataset, desc="Preprocessing")]
print(f"Prepared {len(train_dataset)} samples")

## 6. Load base model + attach LoRA

In [ ]:
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit            = True,
    use_gradient_checkpointing = "unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
)

## 7. Train

In [ ]:
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = PER_DEVICE_BS,
        gradient_accumulation_steps = GRAD_ACCUM,
        max_steps                   = MAX_STEPS,
        warmup_steps                = 10,
        learning_rate               = LEARNING_RATE,
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        remove_unused_columns = False,
        dataset_text_field    = "",
        dataset_kwargs        = {"skip_prepare_dataset": True},
        dataset_num_proc      = 4,
        max_seq_length        = 2048,
    ),
)

trainer_stats = trainer.train()

## 8. Save LoRA adapter

In [ ]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

### Optional: push to Hugging Face Hub

```python
from huggingface_hub import login
login(token="hf_...")
model.push_to_hub("<your-hf-username>/<adapter-name>")
tokenizer.push_to_hub("<your-hf-username>/<adapter-name>")
```